# Create Windowed Datasets for Transfer Learning

## Overview
This notebook creates fixed-length windowed tensors from preprocessed LEMON and migraine EEG data for deep learning.

## Research Protocol
The windowing strategy follows best practices from:
- **Schirrmeister et al. (2017)** - "Deep learning with convolutional neural networks for EEG decoding and visualization" *Human Brain Mapping*
- **Lawhern et al. (2018)** - "EEGNet: a compact convolutional neural network for EEG-based brain–computer interfaces" *Journal of Neural Engineering*
- **Roy et al. (2019)** - "Deep learning-based electroencephalography analysis: a systematic review" *Journal of Neural Engineering*

## Windowing Strategy
- **Window Duration**: 4 seconds (1000 samples at 250 Hz)
  - Captures sufficient information for pattern recognition
  - Standard in EEG deep learning (Craik et al., 2019)
- **Overlap**: 50% (2-second stride)
  - Increases dataset size without introducing bias
  - Common in sequence-based neural networks
- **Artifact Rejection**: Peak-to-peak amplitude threshold (150 µV)
- **Normalization**: Per-channel per-subject z-score
  - Removes inter-subject variability
  - Preserves intra-subject patterns

## Expected Dataset Sizes
- **LEMON**: ~213 subjects × 390 windows = **~83,000 windows**
- **Migraine**: ~31 subjects × 195 windows = **~6,000 windows**
- **Total**: **~89,000 training samples**

In [ ]:
# Import required libraries
import sys
import mne
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import pickle
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.append('src')
from windowed_dataset_builder import WindowedDatasetBuilder

# Set MNE configuration
mne.set_log_level('WARNING')

print(f"MNE Version: {mne.__version__}")
print(f"NumPy Version: {np.__version__}")
print("✓ Libraries loaded successfully")

## 1. Configure Windowing Parameters

### References:
- **Window Duration (4s)**: Schirrmeister et al. (2017) - Optimal for EEG pattern recognition
- **Overlap (50%)**: Krizhevsky et al. (2012) - Data augmentation without label leakage
- **Artifact Threshold (150 µV)**: Delorme et al. (2007) - Standard EEG quality control
- **Z-score Normalization**: LeCun et al. (2012) - Improves neural network convergence

In [ ]:
# Initialize windowed dataset builder
builder = WindowedDatasetBuilder(
    window_duration=4.0,              # 4-second windows
    overlap=0.5,                      # 50% overlap (2-second stride)
    artifact_threshold=150e-6,        # 150 µV peak-to-peak
    min_epochs_per_subject=10,        # Minimum quality threshold
    normalize_per_subject=True,       # Z-score normalization per subject
    verbose=True
)

print("✓ Windowed dataset builder configured")
print(f"  Window duration: {builder.window_duration}s")
print(f"  Overlap: {builder.overlap * 100}%")
print(f"  Stride: {builder.window_duration * (1 - builder.overlap)}s")
print(f"  Artifact threshold: {builder.artifact_threshold * 1e6} µV")
print(f"  Normalization: Per-subject z-score")

## 2. Locate Preprocessed Files

Load all preprocessed EEG files from both LEMON and migraine datasets.

In [ ]:
# Set paths
lemon_preprocessed_dir = Path('data/LEMON_preprocessed')
migraine_preprocessed_dir = Path('data/Migraine_preprocessed')
output_dir = Path('data/windowed_datasets')
output_dir.mkdir(parents=True, exist_ok=True)

# Find all preprocessed files
lemon_files = sorted(list(lemon_preprocessed_dir.glob('*_preprocessed_raw.fif')))
migraine_files = sorted(list(migraine_preprocessed_dir.glob('*_preprocessed_raw.fif')))

print("Preprocessed Files Found:")
print(f"  LEMON: {len(lemon_files)} subjects")
print(f"  Migraine: {len(migraine_files)} subjects")
print(f"  Total: {len(lemon_files) + len(migraine_files)} subjects")

if len(lemon_files) == 0:
    print("\n⚠️ WARNING: No LEMON files found. Run 01_LEMON_Preprocessing_All_Subjects.ipynb first!")

if len(migraine_files) == 0:
    print("\n⚠️ WARNING: No migraine files found. Run 02_Migraine_Preprocessing_All_Subjects.ipynb first!")

## 3. Load Metadata and Labels

Import preprocessing metadata to retrieve subject labels and quality metrics.

In [ ]:
# Load preprocessing metadata
lemon_metadata_file = lemon_preprocessed_dir / 'lemon_preprocessing_metadata.csv'
migraine_metadata_file = migraine_preprocessed_dir / 'migraine_preprocessing_metadata.csv'

# LEMON metadata (unlabeled - for pretraining)
if lemon_metadata_file.exists():
    lemon_metadata = pd.read_csv(lemon_metadata_file)
    print(f"✓ Loaded LEMON metadata: {len(lemon_metadata)} subjects")
else:
    lemon_metadata = pd.DataFrame()
    print("⚠️ No LEMON metadata found")

# Migraine metadata (labeled - for supervised learning)
if migraine_metadata_file.exists():
    migraine_metadata = pd.read_csv(migraine_metadata_file)
    print(f"✓ Loaded migraine metadata: {len(migraine_metadata)} subjects")
    print(f"  - Control: {(migraine_metadata['label'] == 0).sum()} subjects")
    print(f"  - Migraine: {(migraine_metadata['label'] == 1).sum()} subjects")
else:
    migraine_metadata = pd.DataFrame()
    print("⚠️ No migraine metadata found")

## 4. Process LEMON Dataset (Unsupervised Pretraining)

### Purpose:
LEMON data provides healthy brain patterns for unsupervised pretraining, establishing general EEG feature representations.

### Transfer Learning Strategy:
- **Stage 1 (Unsupervised)**: Learn general EEG patterns from LEMON
- **Stage 2 (Supervised)**: Fine-tune on migraine-specific patterns

### References:
- **Self-supervised Learning**: Chen et al. (2020) "A simple framework for contrastive learning of visual representations"
- **EEG Transfer Learning**: Kostas et al. (2021) "BENDR: Using transformers and a contrastive self-supervised learning task to learn from massive amounts of EEG data"

In [ ]:
# Process LEMON subjects
lemon_windows_list = []
lemon_metadata_list = []

if len(lemon_files) > 0:
    print("\n" + "="*70)
    print("PROCESSING LEMON DATASET")
    print("="*70)
    
    for filepath in tqdm(lemon_files, desc="Processing LEMON subjects"):
        subject_id = filepath.stem.replace('_preprocessed_raw', '')
        
        try:
            # Load preprocessed data
            raw = mne.io.read_raw_fif(filepath, preload=True, verbose=False)
            
            # Process subject (no label for unsupervised learning)
            windows, metadata = builder.process_single_subject(
                raw=raw,
                subject_id=subject_id,
                label=None,  # LEMON is unlabeled
                dataset_type='lemon'
            )
            
            lemon_windows_list.append(windows)
            lemon_metadata_list.append(metadata)
            
        except Exception as e:
            print(f"\n✗ Failed: {subject_id} - {str(e)}")
    
    # Combine all LEMON data
    if len(lemon_windows_list) > 0:
        lemon_X = np.vstack(lemon_windows_list)  # (n_windows, n_channels, n_samples)
        lemon_meta = pd.concat(lemon_metadata_list, ignore_index=True)
        
        print(f"\n✓ LEMON Dataset:")
        print(f"  Shape: {lemon_X.shape}")
        print(f"  Memory: {lemon_X.nbytes / 1024**2:.1f} MB")
        print(f"  Subjects: {lemon_meta['subject_id'].nunique()}")
        print(f"  Total Windows: {len(lemon_X)}")
    else:
        lemon_X = None
        lemon_meta = None
        print("\n⚠️ No LEMON data processed")
else:
    lemon_X = None
    lemon_meta = None
    print("\n⚠️ Skipping LEMON (no files found)")

## 5. Process Migraine Dataset (Supervised Learning)

### Purpose:
Migraine data with labels enables supervised classification and personalized treatment.

### Label Encoding:
- **0**: Healthy control
- **1**: Migraine patient

### Clinical Significance:
Labels enable:
- Binary classification (migraine vs control)
- Biomarker identification
- Personalized treatment optimization

### References:
- **Clinical ML Validation**: Varoquaux & Cheplygina (2022) "Machine learning for medical imaging: methodological failures and recommendations for the future"
- **EEG Biomarkers**: Schulz et al. (2016) "Decoding an individual's sensitivity to pain from the multivariate analysis of EEG data"

In [ ]:
# Process migraine subjects
migraine_windows_list = []
migraine_metadata_list = []

if len(migraine_files) > 0:
    print("\n" + "="*70)
    print("PROCESSING MIGRAINE DATASET")
    print("="*70)
    
    for filepath in tqdm(migraine_files, desc="Processing migraine subjects"):
        subject_id = filepath.stem.replace('_preprocessed_raw', '')
        
        # Get label from metadata
        if len(migraine_metadata) > 0:
            subject_meta = migraine_metadata[migraine_metadata['subject_id'] == subject_id]
            if len(subject_meta) > 0:
                label = int(subject_meta['label'].iloc[0])
            else:
                # Infer from subject ID if not in metadata
                label = 0 if subject_id.startswith('C') else 1
        else:
            label = 0 if subject_id.startswith('C') else 1
        
        try:
            # Load preprocessed data
            raw = mne.io.read_raw_fif(filepath, preload=True, verbose=False)
            
            # Process subject with label
            windows, metadata = builder.process_single_subject(
                raw=raw,
                subject_id=subject_id,
                label=label,
                dataset_type='migraine'
            )
            
            migraine_windows_list.append(windows)
            migraine_metadata_list.append(metadata)
            
        except Exception as e:
            print(f"\n✗ Failed: {subject_id} - {str(e)}")
    
    # Combine all migraine data
    if len(migraine_windows_list) > 0:
        migraine_X = np.vstack(migraine_windows_list)  # (n_windows, n_channels, n_samples)
        migraine_meta = pd.concat(migraine_metadata_list, ignore_index=True)
        migraine_y = migraine_meta['label'].values  # Labels for supervised learning
        
        print(f"\n✓ Migraine Dataset:")
        print(f"  Shape: {migraine_X.shape}")
        print(f"  Memory: {migraine_X.nbytes / 1024**2:.1f} MB")
        print(f"  Subjects: {migraine_meta['subject_id'].nunique()}")
        print(f"  Total Windows: {len(migraine_X)}")
        print(f"  Label Distribution:")
        print(f"    Control (0): {(migraine_y == 0).sum()} windows")
        print(f"    Migraine (1): {(migraine_y == 1).sum()} windows")
    else:
        migraine_X = None
        migraine_y = None
        migraine_meta = None
        print("\n⚠️ No migraine data processed")
else:
    migraine_X = None
    migraine_y = None
    migraine_meta = None
    print("\n⚠️ Skipping migraine (no files found)")

## 6. Save Datasets

Save windowed datasets in NumPy format for efficient loading during training.

### File Structure:
- `lemon_windows.npy`: LEMON windowed data (unlabeled)
- `lemon_metadata.csv`: LEMON window metadata
- `migraine_windows.npy`: Migraine windowed data
- `migraine_labels.npy`: Migraine labels (0=control, 1=migraine)
- `migraine_metadata.csv`: Migraine window metadata

### Data Integrity:
- Consistent tensor shape across all windows
- Preserved subject boundaries (no mixing)
- Maintained temporal order within subjects

In [ ]:
print("\n" + "="*70)
print("SAVING WINDOWED DATASETS")
print("="*70)

# Save LEMON dataset
if lemon_X is not None:
    np.save(output_dir / 'lemon_windows.npy', lemon_X)
    lemon_meta.to_csv(output_dir / 'lemon_metadata.csv', index=False)
    print(f"✓ Saved LEMON dataset:")
    print(f"  - {output_dir / 'lemon_windows.npy'}")
    print(f"  - {output_dir / 'lemon_metadata.csv'}")

# Save migraine dataset
if migraine_X is not None:
    np.save(output_dir / 'migraine_windows.npy', migraine_X)
    np.save(output_dir / 'migraine_labels.npy', migraine_y)
    migraine_meta.to_csv(output_dir / 'migraine_metadata.csv', index=False)
    print(f"\n✓ Saved migraine dataset:")
    print(f"  - {output_dir / 'migraine_windows.npy'}")
    print(f"  - {output_dir / 'migraine_labels.npy'}")
    print(f"  - {output_dir / 'migraine_metadata.csv'}")

# Save dataset info
dataset_info = {
    'window_duration': builder.window_duration,
    'overlap': builder.overlap,
    'sampling_rate': 250.0,
    'artifact_threshold': builder.artifact_threshold,
    'normalization': 'per-subject z-score',
    'lemon_subjects': lemon_meta['subject_id'].nunique() if lemon_meta is not None else 0,
    'lemon_windows': len(lemon_X) if lemon_X is not None else 0,
    'migraine_subjects': migraine_meta['subject_id'].nunique() if migraine_meta is not None else 0,
    'migraine_windows': len(migraine_X) if migraine_X is not None else 0,
}

with open(output_dir / 'dataset_info.pkl', 'wb') as f:
    pickle.dump(dataset_info, f)

print(f"\n✓ Saved dataset info: {output_dir / 'dataset_info.pkl'}")
print("="*70)

## 7. Visualize Dataset Statistics

Comprehensive visualization of windowing outcomes and data quality.

In [ ]:
# Create visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Dataset size comparison
datasets = []
sizes = []
if lemon_X is not None:
    datasets.append('LEMON\n(Unlabeled)')
    sizes.append(len(lemon_X))
if migraine_X is not None:
    datasets.append('Migraine\n(Labeled)')
    sizes.append(len(migraine_X))

if len(datasets) > 0:
    axes[0, 0].bar(datasets, sizes, color=['skyblue', 'coral'], edgecolor='black')
    axes[0, 0].set_ylabel('Number of Windows')
    axes[0, 0].set_title('Dataset Size Comparison', fontweight='bold')
    for i, v in enumerate(sizes):
        axes[0, 0].text(i, v + max(sizes)*0.02, f"{v:,}", ha='center', fontweight='bold')

# 2. Windows per subject (LEMON)
if lemon_meta is not None:
    windows_per_subject = lemon_meta.groupby('subject_id').size()
    axes[0, 1].hist(windows_per_subject, bins=30, color='skyblue', edgecolor='black')
    axes[0, 1].axvline(windows_per_subject.mean(), color='red', linestyle='--', 
                       label=f'Mean: {windows_per_subject.mean():.0f}')
    axes[0, 1].set_xlabel('Windows per Subject')
    axes[0, 1].set_ylabel('Number of Subjects')
    axes[0, 1].set_title('LEMON: Windows per Subject', fontweight='bold')
    axes[0, 1].legend()

# 3. Windows per subject (Migraine)
if migraine_meta is not None:
    windows_per_subject_migraine = migraine_meta.groupby('subject_id').size()
    axes[0, 2].hist(windows_per_subject_migraine, bins=20, color='coral', edgecolor='black')
    axes[0, 2].axvline(windows_per_subject_migraine.mean(), color='red', linestyle='--',
                       label=f'Mean: {windows_per_subject_migraine.mean():.0f}')
    axes[0, 2].set_xlabel('Windows per Subject')
    axes[0, 2].set_ylabel('Number of Subjects')
    axes[0, 2].set_title('Migraine: Windows per Subject', fontweight='bold')
    axes[0, 2].legend()

# 4. Label distribution (Migraine)
if migraine_y is not None:
    label_counts = pd.Series(migraine_y).value_counts().sort_index()
    axes[1, 0].bar(['Control (0)', 'Migraine (1)'], label_counts.values, 
                   color=['lightgreen', 'lightcoral'], edgecolor='black')
    axes[1, 0].set_ylabel('Number of Windows')
    axes[1, 0].set_title('Migraine Dataset: Label Distribution', fontweight='bold')
    for i, v in enumerate(label_counts.values):
        axes[1, 0].text(i, v + max(label_counts)*0.02, f"{v:,}", ha='center', fontweight='bold')

# 5. Sample EEG window visualization (LEMON)
if lemon_X is not None:
    sample_window = lemon_X[0, :5, :]  # First 5 channels
    time_axis = np.arange(sample_window.shape[1]) / 250.0  # Convert to seconds
    for ch_idx in range(sample_window.shape[0]):
        axes[1, 1].plot(time_axis, sample_window[ch_idx, :] + ch_idx * 5, linewidth=0.5)
    axes[1, 1].set_xlabel('Time (s)')
    axes[1, 1].set_ylabel('Channel (offset for visibility)')
    axes[1, 1].set_title('Sample LEMON Window (5 channels)', fontweight='bold')
    axes[1, 1].grid(True, alpha=0.3)

# 6. Sample EEG window visualization (Migraine)
if migraine_X is not None:
    sample_window_m = migraine_X[0, :5, :]  # First 5 channels
    time_axis_m = np.arange(sample_window_m.shape[1]) / 250.0
    for ch_idx in range(sample_window_m.shape[0]):
        axes[1, 2].plot(time_axis_m, sample_window_m[ch_idx, :] + ch_idx * 5, linewidth=0.5)
    axes[1, 2].set_xlabel('Time (s)')
    axes[1, 2].set_ylabel('Channel (offset for visibility)')
    axes[1, 2].set_title('Sample Migraine Window (5 channels)', fontweight='bold')
    axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'windowed_dataset_statistics.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Dataset statistics visualization saved")

## 8. Summary Report

Comprehensive summary of windowing outcomes.

In [ ]:
print("\n" + "="*70)
print("WINDOWED DATASET CREATION SUMMARY")
print("="*70)

print("\nConfiguration:")
print("-" * 70)
print(f"  Window duration: {builder.window_duration}s ({int(builder.window_duration * 250)} samples)")
print(f"  Overlap: {builder.overlap * 100}% ({builder.window_duration * builder.overlap}s)")
print(f"  Stride: {builder.window_duration * (1 - builder.overlap)}s")
print(f"  Artifact threshold: {builder.artifact_threshold * 1e6} µV")
print(f"  Normalization: Per-subject z-score")

if lemon_X is not None:
    print("\nLEMON Dataset (Unsupervised Pretraining):")
    print("-" * 70)
    print(f"  Subjects processed: {lemon_meta['subject_id'].nunique()}")
    print(f"  Total windows: {len(lemon_X):,}")
    print(f"  Tensor shape: {lemon_X.shape}")
    print(f"  Memory size: {lemon_X.nbytes / 1024**2:.1f} MB")
    print(f"  Avg windows/subject: {len(lemon_X) / lemon_meta['subject_id'].nunique():.0f}")

if migraine_X is not None:
    print("\nMigraine Dataset (Supervised Learning):")
    print("-" * 70)
    print(f"  Subjects processed: {migraine_meta['subject_id'].nunique()}")
    print(f"  Total windows: {len(migraine_X):,}")
    print(f"  Tensor shape: {migraine_X.shape}")
    print(f"  Memory size: {migraine_X.nbytes / 1024**2:.1f} MB")
    print(f"  Avg windows/subject: {len(migraine_X) / migraine_meta['subject_id'].nunique():.0f}")
    print(f"\n  Label Distribution:")
    print(f"    Control (0): {(migraine_y == 0).sum():,} windows ({(migraine_y == 0).sum() / len(migraine_y) * 100:.1f}%)")
    print(f"    Migraine (1): {(migraine_y == 1).sum():,} windows ({(migraine_y == 1).sum() / len(migraine_y) * 100:.1f}%)")

if lemon_X is not None and migraine_X is not None:
    total_windows = len(lemon_X) + len(migraine_X)
    total_subjects = lemon_meta['subject_id'].nunique() + migraine_meta['subject_id'].nunique()
    print("\nCombined Dataset:")
    print("-" * 70)
    print(f"  Total subjects: {total_subjects}")
    print(f"  Total windows: {total_windows:,}")
    print(f"  Total size: {(lemon_X.nbytes + migraine_X.nbytes) / 1024**2:.1f} MB")

print("\n" + "="*70)
print("✓ Windowed datasets created successfully!")
print(f"✓ Datasets saved to: {output_dir}")
print("="*70)

## 9. Verify Data Integrity

Sanity checks to ensure data quality before training.

In [ ]:
print("\nData Integrity Checks:")
print("="*70)

# Check for NaN or Inf values
if lemon_X is not None:
    has_nan = np.isnan(lemon_X).any()
    has_inf = np.isinf(lemon_X).any()
    print(f"LEMON - NaN values: {'❌ FOUND' if has_nan else '✓ None'}")
    print(f"LEMON - Inf values: {'❌ FOUND' if has_inf else '✓ None'}")
    print(f"LEMON - Value range: [{lemon_X.min():.2f}, {lemon_X.max():.2f}]")

if migraine_X is not None:
    has_nan_m = np.isnan(migraine_X).any()
    has_inf_m = np.isinf(migraine_X).any()
    print(f"\nMigraine - NaN values: {'❌ FOUND' if has_nan_m else '✓ None'}")
    print(f"Migraine - Inf values: {'❌ FOUND' if has_inf_m else '✓ None'}")
    print(f"Migraine - Value range: [{migraine_X.min():.2f}, {migraine_X.max():.2f}]")

# Verify normalization (should be approximately N(0,1))
if lemon_X is not None:
    lemon_mean = np.mean(lemon_X)
    lemon_std = np.std(lemon_X)
    print(f"\nLEMON - Global mean: {lemon_mean:.4f} (should be ~0)")
    print(f"LEMON - Global std: {lemon_std:.4f} (should be ~1)")

if migraine_X is not None:
    migraine_mean = np.mean(migraine_X)
    migraine_std = np.std(migraine_X)
    print(f"\nMigraine - Global mean: {migraine_mean:.4f} (should be ~0)")
    print(f"Migraine - Global std: {migraine_std:.4f} (should be ~1)")

print("\n" + "="*70)
print("✓ Data integrity verified!")
print("="*70)

## 10. Next Steps

1. ✅ LEMON preprocessing complete
2. ✅ Migraine preprocessing complete
3. ✅ **Windowed datasets created** (this notebook)
4. ⏭️ Run `04_Transfer_Learning_Training.ipynb` to train the model

---

## References

1. Chen, T., et al. (2020). "A simple framework for contrastive learning of visual representations." *International Conference on Machine Learning*, 1597-1607.

2. Craik, A., et al. (2019). "Deep learning for electroencephalogram (EEG) classification tasks: a review." *Journal of Neural Engineering* 16(3), 031001.

3. Delorme, A., et al. (2007). "Enhanced detection of artifacts in EEG data using higher-order statistics and independent component analysis." *NeuroImage* 34(4), 1443-1449.

4. Kostas, D., et al. (2021). "BENDR: Using transformers and a contrastive self-supervised learning task to learn from massive amounts of EEG data." *Frontiers in Human Neuroscience* 15, 653659.

5. Krizhevsky, A., et al. (2012). "ImageNet classification with deep convolutional neural networks." *Advances in Neural Information Processing Systems* 25, 1097-1105.

6. Lawhern, V. J., et al. (2018). "EEGNet: a compact convolutional neural network for EEG-based brain–computer interfaces." *Journal of Neural Engineering* 15(5), 056013.

7. LeCun, Y., et al. (2012). "Efficient BackProp." *Neural Networks: Tricks of the Trade*, Springer, 9-48.

8. Roy, Y., et al. (2019). "Deep learning-based electroencephalography analysis: a systematic review." *Journal of Neural Engineering* 16(5), 051001.

9. Schirrmeister, R. T., et al. (2017). "Deep learning with convolutional neural networks for EEG decoding and visualization." *Human Brain Mapping* 38(11), 5391-5420.

10. Schulz, E., et al. (2016). "Decoding an individual's sensitivity to pain from the multivariate analysis of EEG data." *Cerebral Cortex* 26(4), 1722-1734.

11. Varoquaux, G., & Cheplygina, V. (2022). "Machine learning for medical imaging: methodological failures and recommendations for the future." *NPJ Digital Medicine* 5(1), 48.